In [4]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog
import numpy as np
import pandas as pd

STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 17

Path("output").mkdir(exist_ok=True)

# --- FILE SELECTOR DIALOG ---
print("Please select your 'secom.data' file in the popup window...")
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

data_file = filedialog.askopenfilename(
    title="Select secom.data", filetypes=[("Data files", "*.data"), ("All files", "*.*")]
)
if not data_file:
  raise ValueError("No data file selected!")

print(f"Please select your 'secom_labels.data' file in the popup window...")
labels_file = filedialog.askopenfilename(
    title="Select secom_labels.data",
    filetypes=[("Data files", "*.data"), ("All files", "*.*")],
)
if not labels_file:
  raise ValueError("No labels file selected!")

# Load Data
X_raw = pd.read_csv(data_file, sep=r"\s+", header=None)
y_raw = pd.read_csv(labels_file, sep=r"\s+", header=None)
df = X_raw.copy()
df["Outcome"] = y_raw.iloc[:, 0]
X = df.iloc[:, :-1]

# --- DYNAMIC CALCULATIONS ---
# 1. Selected features (first 5 numeric columns)
selected_feature_indices = list(X.select_dtypes(include=[np.number]).columns[:5])

# 2. Missing percentage per feature (Vectorized)
missing_pct = X.isnull().mean() * 100
highest_missing_feat = missing_pct.idxmax()
highest_missing_val = missing_pct.max()

# 3. Flagged observations (Vectorized row-wise check)
row_missing_pct = X.isnull().mean(axis=1) * 100
flagged_mask = row_missing_pct > 20.0
total_flagged = int(flagged_mask.sum())
highest_flagged_feat = (
    X.loc[flagged_mask].isnull().mean().idxmax()
    if total_flagged > 0
    else highest_missing_feat
)

# 4. Dynamic Verification Check: Ensure vectorized outputs are valid and non-empty
vectorized_verified = bool(
    isinstance(missing_pct, pd.Series)
    and not missing_pct.empty
    and not pd.isna(highest_missing_val)
)

# Summary dataframe generation
summary_df = pd.DataFrame({
    "Feature": X.columns,
    "Missing_Pct": missing_pct,
})

# Export results
summary_df.to_csv("output/measurement_quality.csv", index=False)

print("\n--- Exercise 1 Assessment Data (Fully Dynamic) ---")
print(f"Selected Features: {selected_feature_indices}")
print(
    f"Highest Missing Percentage: Feature {highest_missing_feat}"
    f" ({highest_missing_val:.2f}%)"
)
print(f"Feature with Highest Flagged Percentage: {highest_flagged_feat}")
print(f"Total Flagged Observations: {total_flagged}")
print(f"Vectorized Results Verified: {vectorized_verified}")

Please select your 'secom.data' file in the popup window...
Please select your 'secom_labels.data' file in the popup window...

--- Exercise 1 Assessment Data (Fully Dynamic) ---
Selected Features: [0, 1, 2, 3, 4]
Highest Missing Percentage: Feature 157 (91.19%)
Feature with Highest Flagged Percentage: 13
Total Flagged Observations: 3
Vectorized Results Verified: True
